# What influences car MPG? A CausalIF demo

This notebook uses the [awslabs/causalif](https://github.com/awslabs/causalif) framework to explore the causal drivers of fuel economy (MPG) in the classic [Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg) dataset.

CausalIF combines an LLM's background knowledge with Bayesian structure learning (Hill Climbing + BDeu, plus bootstrap stability) to move from raw observational data to a directed causal graph. The LLM is served through **Amazon Bedrock**.

**Runtime target:** AWS SageMaker Studio.

This first section just gets the environment ready: install the packages, import them, and set up a configurable AWS region.

## 1. Install packages

Install the packages directly with the notebook `%pip` magic so they land in the active kernel. SageMaker Studio images already ship `pandas`, `numpy`, `plotly`, and `pgmpy`; `causalif` reuses those and pulls in anything missing. `langchain-aws` provides the Bedrock-backed LLM (`ChatBedrockConverse`) and the Knowledge Base retriever.

In [ ]:
# CausalIF demo - Python packages
# SageMaker Studio images already ship pandas / numpy / plotly / pgmpy,
# but pinning causalif here keeps the demo reproducible across kernels.
# langchain-aws>=1.6 is required for managed Knowledge Base retrieval
# (managedSearchConfiguration in Section 5); older versions only support
# vectorSearchConfiguration for customer-managed KBs.
%pip install causalif==0.1.10 "langchain-aws>=1.6"

## 2. Import the main packages

The core CausalIF entry points we'll use:

- `set_causalif_engine` — configures the engine (LLM, dataframe, domains, etc.)
- `causalif` — runs a natural-language causal query
- `visualize_causalif_results` — renders the interactive Plotly causal graph
- `causalif_intervene` — asks interventional "what if" (do-operator) questions

`ChatBedrockConverse` (from `langchain-aws`) is the Bedrock-backed LLM CausalIF reasons with.

In [ ]:
import pandas as pd
import numpy as np

from causalif import (
    set_causalif_engine,
    causalif,
    visualize_causalif_results,
    causalif_intervene,
)
from langchain_aws import ChatBedrockConverse

print("Imports ready.")

## 3. Region and model configuration

Anything region- or model-specific lives in one place so it's easy to change. Defaults target **us-west-2**. Serverless Bedrock models auto-enable on first use, so there's no manual model-access step; just make sure your execution role can invoke the model (`bedrock:InvokeModel`) and isn't blocked by an IAM policy or SCP. For Anthropic Claude, a first-time user may be asked to submit brief use-case details before the first call.

In [ ]:
# --- Configurable settings ---------------------------------------------------
AWS_REGION = "us-west-2"

# Bedrock model id used by CausalIF for causal reasoning.
# Swap this for any Bedrock model you have access to in AWS_REGION. Use a
# CURRENT model - older ones (e.g. Claude Sonnet 4, ...-4-20250514) are marked
# Legacy by the provider and Converse fails with 'ResourceNotFoundException:
# ... Access denied. This Model is marked by provider as Legacy'.
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# (Optional) Amazon Bedrock Knowledge Base ID for RAG-grounded causal reasoning.
# Leave as None to run on the LLM's background knowledge alone (no retriever).
# To create a Knowledge Base and get this ID, follow Step 5 in the user guide,
# then paste the ID here, e.g. KNOWLEDGE_BASE_ID = "ABCD1234EF".
KNOWLEDGE_BASE_ID = None

# Path to the observational data (sitting in the workspace root).
DATA_PATH = "auto-mpg.data"
# -----------------------------------------------------------------------------

print(f"Region: {AWS_REGION}")
print(f"Model:  {BEDROCK_MODEL_ID}")
print(f"Knowledge Base ID: {KNOWLEDGE_BASE_ID or '(none - using background knowledge only)'}")

In [ ]:
# Initialise the Bedrock-backed LLM. temperature=0.0 keeps causal reasoning stable across runs.
model = ChatBedrockConverse(
    model_id=BEDROCK_MODEL_ID,
    temperature=0.0,
    region_name=AWS_REGION,
)

print("Bedrock model client created.")

## 4. Prepare the data

**Getting the data:** download the Auto MPG dataset from the UCI Machine Learning Repository: [auto+mpg.zip](https://archive.ics.uci.edu/static/public/9/auto+mpg.zip). Unzip it, extract the `auto-mpg.data` file, and place it in the **root (home) of this workspace** — the same folder as this notebook. That's the path `DATA_PATH` points to.

The Auto MPG file is whitespace-delimited with no header. The columns, in order, are:

`mpg`, `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration`, `model_year`, `origin`, `car_name`.

A few things to sort out before handing the data to CausalIF:

- **`horsepower`** uses `?` for missing values, so we read those as `NaN` and drop the affected rows.
- **`car_name`** is free text (e.g. "ford torino") — not a usable causal factor, so we exclude it.
- **`origin`** is a number, but it's really a *category* encoding the region of origin (1 = USA, 2 = Europe, 3 = Japan). Treating it as a continuous factor would be misleading, so we exclude it too.

That leaves seven numeric factors for the causal analysis, with `mpg` as the target.

In [ ]:
# Column names for the headerless Auto MPG file, in file order.
COLUMN_NAMES = [
    "mpg",
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
    "origin",
    "car_name",
]

# Whitespace-delimited; '?' marks missing horsepower values.
raw_df = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    names=COLUMN_NAMES,
    na_values="?",
)

print(f"Loaded {len(raw_df)} rows, {raw_df.shape[1]} columns.")
raw_df.head()

In [ ]:
# Factors we do NOT want CausalIF to analyse:
#   car_name - free-text label, not a causal factor
#   origin   - numeric but really a categorical region code (1=USA, 2=Europe, 3=Japan)
EXCLUDED_COLUMNS = ["origin", "car_name"]

# Keep only the analysis factors, then drop rows with missing values (the '?' horsepower entries).
df = raw_df.drop(columns=EXCLUDED_COLUMNS).dropna().reset_index(drop=True)

FACTOR_COLUMNS = list(df.columns)
print(f"Analysis factors ({len(FACTOR_COLUMNS)}): {FACTOR_COLUMNS}")
print(f"Rows after cleaning: {len(df)}")
df.head()

### Factor descriptions

CausalIF reasons about causal direction using the *meaning* of each column, not just its name. Passing a `factor_descriptions` string (Markdown) with a plain-English definition of each factor gives the LLM the domain context it needs to orient edges correctly. We describe exactly the seven factors we're analysing.

In [ ]:
factor_descriptions = """# Factor Definitions
- mpg: fuel efficiency of the car in miles per gallon (higher means more efficient)
- cylinders: number of cylinders in the engine
- displacement: total engine displacement in cubic inches (engine size)
- horsepower: engine power output in horsepower
- weight: vehicle weight in pounds
- acceleration: time in seconds to accelerate from 0 to 60 mph (lower means faster acceleration)
- model_year: model year of the car (e.g. 70 = 1970)
"""

print(factor_descriptions)

## 5. (Optional) Build a Knowledge Base retriever for RAG

CausalIF works well on the LLM's background knowledge alone, which is the default here. You can optionally ground its causal reasoning in your own domain documents through an Amazon Bedrock Knowledge Base (KB): during edge analysis, CausalIF retrieves relevant passages to inform each association vote.

To use a KB, follow **Step 5 in the user guide** (`USER-GUIDE.md`) — it walks through creating an S3 bucket, uploading the reference documents, and creating the Knowledge Base — then paste the resulting **Knowledge Base ID** into `KNOWLEDGE_BASE_ID` in the configuration cell (Section 3).

The cell below builds the retriever **only if** `KNOWLEDGE_BASE_ID` is set. If it's `None`, the retriever is skipped and the notebook runs on background knowledge alone — no other changes needed.

In [ ]:
# Build a Knowledge Base retriever only when KNOWLEDGE_BASE_ID is set.
# retriever stays None otherwise, and Section 6 simply passes None (no RAG).
retriever = None

if KNOWLEDGE_BASE_ID:
    from langchain_aws.retrievers import AmazonKnowledgeBasesRetriever

    retriever = AmazonKnowledgeBasesRetriever(
        knowledge_base_id=KNOWLEDGE_BASE_ID,
        region_name=AWS_REGION,
        retrieval_config={"managedSearchConfiguration": {"numberOfResults": 20}},
    )
    print(f"Retriever created for Knowledge Base '{KNOWLEDGE_BASE_ID}'.")
else:
    print("No KNOWLEDGE_BASE_ID set - skipping retriever (using background knowledge only).")

## 6. Configure the CausalIF engine

`set_causalif_engine` wires everything together: the Bedrock model, our cleaned dataframe, the factor descriptions, and some analysis settings.

Key choices for this demo:

- **`domains`** — gives the LLM the right background knowledge to reason with. For cars that's the automotive / mechanical engineering space. The README treats this as effectively mandatory.
- **`enable_causal_estimate=True`** — turns on causal effect estimation (ATE) so we can later ask interventional "what if" questions with `causalif_intervene`.
- **`bootstrap_iterations` / `bootstrap_threshold`** — resample the data 50 times and keep only edges whose direction is stable in ≥70% of resamples. These are the framework's benchmark defaults.
- **`retriever_tool=retriever`** — the optional KB retriever from Section 5. It's `None` unless you set `KNOWLEDGE_BASE_ID`, in which case CausalIF stays on the LLM's background knowledge plus the observational data (fine for a well-understood domain like cars).

In [ ]:
set_causalif_engine(
    model=model,
    dataframe=df,
    domains=["automotive_engineering", "mechanical_engineering", "vehicle_fuel_economy"],
    factor_descriptions=factor_descriptions,
    selected_dataframe_columns=FACTOR_COLUMNS,
    retriever_tool=retriever,
    enable_causal_estimate=True,
    bootstrap_iterations=50,
    bootstrap_threshold=0.5,
    max_parallel_queries=50,
)

print("CausalIF engine configured.")

## 7. Run the causal analysis

Now we ask the actual question. CausalIF understands a few natural-language query formats; the `what influences <target_factor>` form maps cleanly onto our question. The target factor must be an exact column name — here, `mpg`.

This kicks off the full pipeline: LLM edge voting to build the prior, Bayesian orientation with bootstrap stability, and causal effect estimation. It makes several Bedrock calls per factor pair, so expect it to take a little while.

In [ ]:
result = causalif("what influences mpg")

print(result["summary"])

## 8. Visualise the causal graph

`visualize_causalif_results` renders the discovered causal graph as an interactive Plotly figure. Nodes are coloured by their degree of separation from the target (`mpg`), arrows show causal direction, and hovering over edges reveals the estimated effects.

In [ ]:
fig = visualize_causalif_results(result)
fig.show()

## 9. Save the result

Write the full result dictionary to `result.json` for later inspection. The result can hold numpy values and tuples (e.g. graph edges), so we pass `default=str` to serialise anything that isn't natively JSON-friendly.

In [ ]:
import json

with open("result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, default=str)

print("Wrote result.json")